In [ ]:
!pip -q install ninja transformers accelerate

import os, math, json, time, torch, torch.nn.functional as F
from torch.utils.cpp_extension import load

assert torch.cuda.is_available(), "Switch runtime to A100 GPU."
dev = "cuda"
print("GPU     :", torch.cuda.get_device_name(0))
print("torch   :", torch.__version__)
import transformers; print("transformers:", transformers.__version__)
for d in ["kernels", "benchmarks", "src", "tests", "plots"]:
    os.makedirs(d, exist_ok=True)
print("dirs ready")

# from huggingface_hub import login; login()  # paste your HF token when prompted
MODEL = "unsloth/Llama-3.2-1B"
print("model   :", MODEL)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.4 MB/s eta 0:00:00


In [ ]:

fp16_src = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>
template <int D, int W>
__global__ void decode_attn_fp16_mw_kernel(
    const __half* __restrict__ q, const __half* __restrict__ K, const __half* __restrict__ V,
    __half* __restrict__ o, int N, float scale){
    const int head=blockIdx.x, lane=threadIdx.x&31, warp=threadIdx.x>>5, ELEMS=D/32;
    const __half* qh=q+(size_t)head*D; const __half* Kh=K+(size_t)head*N*D;
    const __half* Vh=V+(size_t)head*N*D; __half* oh=o+(size_t)head*D;
    float q_reg[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) q_reg[e]=__half2float(qh[lane+e*32]);
    float m_run=-INFINITY,l_run=0.0f,o_run[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_run[e]=0.0f;
    for(int i=warp;i<N;i+=W){
        const __half* Ki=Kh+(size_t)i*D; float partial=0.0f;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) partial+=q_reg[e]*__half2float(Ki[lane+e*32]);
        #pragma unroll
        for(int off=16;off>0;off>>=1) partial+=__shfl_down_sync(0xffffffff,partial,off);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;
        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p; const __half* Vi=Vh+(size_t)i*D;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_run[e]=beta*o_run[e]+p*__half2float(Vi[lane+e*32]);
        m_run=m_new;
    }
    __shared__ float sm[W],sl[W],so[W][D];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    #pragma unroll
    for(int e=0;e<ELEMS;++e) so[warp][lane+e*32]=o_run[e];
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float m_w=sm[w],m_new=fmaxf(m_c,m_w);
            float a=__expf(m_c-m_new),b=__expf(m_w-m_new); l_c=a*l_c+b*sl[w];
            #pragma unroll
            for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b*so[w][lane+e*32]; m_c=m_new; }
        #pragma unroll
        for(int e=0;e<ELEMS;++e) oh[lane+e*32]=__float2half(o_c[e]/l_c);
    }
}
torch::Tensor decode_attn_fp16_mw(torch::Tensor q, torch::Tensor K, torch::Tensor V, int64_t W){
    TORCH_CHECK(q.is_cuda()&&K.is_cuda()&&V.is_cuda(),"cuda");
    TORCH_CHECK(q.scalar_type()==torch::kHalf&&K.scalar_type()==torch::kHalf&&V.scalar_type()==torch::kHalf,"fp16");
    TORCH_CHECK(q.is_contiguous()&&K.is_contiguous()&&V.is_contiguous(),"contig");
    const int B=K.size(0),N=K.size(1),D=K.size(2);
    auto o=torch::empty({B,D},q.options()); const float scale=1.0f/sqrtf((float)D);
    dim3 grid(B),block(32*W);
    #define L(Dv,Wv) decode_attn_fp16_mw_kernel<Dv,Wv><<<grid,block>>>((const __half*)q.data_ptr<at::Half>(),(const __half*)K.data_ptr<at::Half>(),(const __half*)V.data_ptr<at::Half>(),(__half*)o.data_ptr<at::Half>(),N,scale)
    TORCH_CHECK(D==64||D==128,"D");
    if(D==64){switch(W){case 2:L(64,2);break;case 4:L(64,4);break;case 8:L(64,8);break;default:TORCH_CHECK(false,"W");}}
    else{switch(W){case 2:L(128,2);break;case 4:L(128,4);break;case 8:L(128,8);break;default:TORCH_CHECK(false,"W");}}
    #undef L
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"launch"); return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_fp16_mw",&decode_attn_fp16_mw,"fp16 fused");}
"""
open("kernels/decode_attn_fp16_mw.cu","w").write(fp16_src)

int8_src = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>
template <int D, int W>
__global__ void decode_attn_int8_mw_kernel(
    const __half* __restrict__ q, const int8_t* __restrict__ K, const int8_t* __restrict__ V,
    const __half* __restrict__ Ks, const __half* __restrict__ Vs, __half* __restrict__ o,
    int N, float scale){
    const int head=blockIdx.x, lane=threadIdx.x&31, warp=threadIdx.x>>5, ELEMS=D/32;
    const __half* qh=q+(size_t)head*D; const int8_t* Kh=K+(size_t)head*N*D;
    const int8_t* Vh=V+(size_t)head*N*D; const __half* Ksh=Ks+(size_t)head*N;
    const __half* Vsh=Vs+(size_t)head*N; __half* oh=o+(size_t)head*D;
    float q_reg[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) q_reg[e]=__half2float(qh[lane+e*32]);
    float m_run=-INFINITY,l_run=0.0f,o_run[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_run[e]=0.0f;
    for(int i=warp;i<N;i+=W){
        const int8_t* Ki=Kh+(size_t)i*D; float ksc=__half2float(Ksh[i]); float partial=0.0f;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) partial+=q_reg[e]*((float)Ki[lane+e*32]*ksc);
        #pragma unroll
        for(int off=16;off>0;off>>=1) partial+=__shfl_down_sync(0xffffffff,partial,off);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;
        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p; const int8_t* Vi=Vh+(size_t)i*D; float vsc=__half2float(Vsh[i]);
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_run[e]=beta*o_run[e]+p*((float)Vi[lane+e*32]*vsc);
        m_run=m_new;
    }
    __shared__ float sm[W],sl[W],so[W][D];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    #pragma unroll
    for(int e=0;e<ELEMS;++e) so[warp][lane+e*32]=o_run[e];
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float m_w=sm[w],m_new=fmaxf(m_c,m_w);
            float a=__expf(m_c-m_new),b=__expf(m_w-m_new); l_c=a*l_c+b*sl[w];
            #pragma unroll
            for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b*so[w][lane+e*32]; m_c=m_new; }
        #pragma unroll
        for(int e=0;e<ELEMS;++e) oh[lane+e*32]=__float2half(o_c[e]/l_c);
    }
}
torch::Tensor decode_attn_int8_mw(torch::Tensor q, torch::Tensor K, torch::Tensor V,
    torch::Tensor Ks, torch::Tensor Vs, int64_t W){
    const int B=K.size(0),N=K.size(1),D=K.size(2);
    TORCH_CHECK(K.scalar_type()==torch::kChar&&V.scalar_type()==torch::kChar,"int8");
    auto o=torch::empty({B,D},q.options()); const float scale=1.0f/sqrtf((float)D);
    dim3 grid(B),block(32*W);
    #define L(Dv,Wv) decode_attn_int8_mw_kernel<Dv,Wv><<<grid,block>>>((const __half*)q.data_ptr<at::Half>(),(const int8_t*)K.data_ptr<int8_t>(),(const int8_t*)V.data_ptr<int8_t>(),(const __half*)Ks.data_ptr<at::Half>(),(const __half*)Vs.data_ptr<at::Half>(),(__half*)o.data_ptr<at::Half>(),N,scale)
    TORCH_CHECK(D==64||D==128,"D");
    if(D==64){switch(W){case 2:L(64,2);break;case 4:L(64,4);break;case 8:L(64,8);break;default:TORCH_CHECK(false,"W");}}
    else{switch(W){case 2:L(128,2);break;case 4:L(128,4);break;case 8:L(128,8);break;default:TORCH_CHECK(false,"W");}}
    #undef L
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"launch"); return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_int8_mw",&decode_attn_int8_mw,"int8 fused");}
"""
open("kernels/decode_attn_int8_mw.cu","w").write(int8_src)

mod_fp16 = load(name="decode_attn_fp16_mw", sources=["kernels/decode_attn_fp16_mw.cu"], extra_cuda_cflags=["-O3"], verbose=True)
mod_int8 = load(name="decode_attn_int8_mw", sources=["kernels/decode_attn_int8_mw.cu"], extra_cuda_cflags=["-O3"], verbose=True)
print("fp16 + int8 kernels compiled OK")


Re-verify fp16 + int8 correctness

In [ ]:

torch.manual_seed(0)
def sdpa_ref(q,K,V):
    B,D=q.shape; N=K.shape[1]
    return F.scaled_dot_product_attention(q.view(B,1,1,D),K.view(B,1,N,D),V.view(B,1,N,D)).view(B,D)
def qpt(X):  # per-token symmetric int8
    a=X.abs().amax(2,keepdim=True); s=(a/127.0).clamp(min=1e-8)
    return torch.clamp(torch.round(X/s),-127,127).to(torch.int8).contiguous(), s.squeeze(2).half().contiguous()

ok=True
for W in [2,4,8]:
    for (B,N) in [(1,1024),(8,2048),(256,4096)]:
        q=torch.randn(B,64,dtype=torch.float16,device=dev).contiguous()
        K=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
        V=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
        o16=mod_fp16.decode_attn_fp16_mw(q,K,V,W); r=sdpa_ref(q,K,V)
        p16=torch.allclose(o16,r,atol=2e-2)
        Kq,Ks=qpt(K); Vq,Vs=qpt(V)
        o8=mod_int8.decode_attn_int8_mw(q,Kq,Vq,Ks,Vs,W)
        Kd=(Kq.float()*Ks.float()[:,:,None]).half(); Vd=(Vq.float()*Vs.float()[:,:,None]).half()
        r8=F.scaled_dot_product_attention(q.view(B,1,1,64),Kd.view(B,1,N,64),Vd.view(B,1,N,64)).view(B,64)
        p8=torch.allclose(o8,r8,atol=3e-2)
        ok&=p16&p8
        print(f"W={W} B={B:4d} N={N:5d} | fp16 {'PASS' if p16 else 'FAIL'} | int8 {'PASS' if p8 else 'FAIL'}")
print("\nALL PASS" if ok else "\nFAILED — stop")
assert ok, "kernel correctness regressed on fresh runtime"


W=2 B=   1 N= 1024 | fp16 PASS | int8 PASS
W=2 B=   8 N= 2048 | fp16 PASS | int8 PASS
W=2 B= 256 N= 4096 | fp16 PASS | int8 PASS
W=4 B=   1 N= 1024 | fp16 PASS | int8 PASS
W=4 B=   8 N= 2048 | fp16 PASS | int8 PASS
W=4 B= 256 N= 4096 | fp16 PASS | int8 PASS
W=8 B=   1 N= 1024 | fp16 PASS | int8 PASS
W=8 B=   8 N= 2048 | fp16 PASS | int8 PASS
W=8 B= 256 N= 4096 | fp16 PASS | int8 PASS

ALL PASS


int4 KV quantization

In [ ]:

torch.manual_seed(0)
def quant_int4_per_token(X):            # X:[B,N,D] fp16 -> packed uint8 [B,N,D/2], scale fp16 [B,N]
    B,N,D=X.shape
    a=X.abs().amax(2,keepdim=True); s=(a/7.0).clamp(min=1e-8)
    q=torch.clamp(torch.round(X/s),-7,7).to(torch.int8)        # [-7,7]
    q=q.view(B,N,D//2,2)
    lo=(q[...,0] & 0xF); hi=(q[...,1] & 0xF)                   # 4-bit two's complement in a nibble
    packed=(lo | (hi<<4)).to(torch.uint8)                      # byte t holds elems (2t, 2t+1)
    return packed.contiguous(), s.squeeze(2).half().contiguous()

def dequant_int4(packed, s):            # inverse, for the reference
    B,N,H=packed.shape; D=H*2
    b=packed.to(torch.int16)
    lo=b & 0xF; hi=(b>>4)&0xF
    lo=torch.where(lo>=8, lo-16, lo); hi=torch.where(hi>=8, hi-16, hi)
    q=torch.stack([lo,hi],dim=-1).view(B,N,D).to(torch.float32)
    return (q*s.float()[:,:,None])

B,N,D=2,1024,64
X=torch.randn(B,N,D,dtype=torch.float16)
packed,s=quant_int4_per_token(X)
deq=dequant_int4(packed,s)
# round-trip through explicit quant to compare (dequant should match quantized values exactly)
a=X.abs().amax(2,keepdim=True); sref=(a/7.0).clamp(min=1e-8)
qref=torch.clamp(torch.round(X/sref),-7,7).float()*sref.float()
print("pack/unpack exact:", torch.allclose(deq, qref, atol=1e-3))
rel=(deq-X.float()).abs().max().item()/X.float().abs().max().item()
print(f"int4 dequant vs original: rel {rel:.4e}  (expect larger than int8's ~1%)")
assert torch.allclose(deq,qref,atol=1e-3), "int4 pack/unpack byte order WRONG"
print("int4 pack/unpack proven in Python")


pack/unpack exact: True
int4 dequant vs original: rel 7.1036e-02  (expect larger than int8's ~1%)
int4 pack/unpack proven in Python


int4 kernel + compile

Layout mirrors the proven 2-elements-per-lane pattern: lane t reads packed byte t, unpacks
nibbles, 2 dequant'd elements (2t, 2t+1). head_dim=64, 32 bytes/token, 32 lanes

In [ ]:

int4_src = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>
// int4 KV (packed 2/byte), per-token symmetric scales, multi-warp. D=64 only.
// lane t owns byte t -> elems (2t lo-nibble, 2t+1 hi-nibble).
template <int W>
__global__ void decode_attn_int4_kernel(
    const __half* __restrict__ q, const uint8_t* __restrict__ K, const uint8_t* __restrict__ V,
    const __half* __restrict__ Ks, const __half* __restrict__ Vs, __half* __restrict__ o,
    int N, float scale){
    const int D=64, H=32; // H = packed bytes per token = D/2
    const int head=blockIdx.x, lane=threadIdx.x&31, warp=threadIdx.x>>5;
    const __half* qh=q+(size_t)head*D; const uint8_t* Kh=K+(size_t)head*N*H;
    const uint8_t* Vh=V+(size_t)head*N*H; const __half* Ksh=Ks+(size_t)head*N;
    const __half* Vsh=Vs+(size_t)head*N; __half* oh=o+(size_t)head*D;
    float q0=__half2float(qh[2*lane+0]), q1=__half2float(qh[2*lane+1]);
    float m_run=-INFINITY,l_run=0.0f,o0=0.0f,o1=0.0f;
    for(int i=warp;i<N;i+=W){
        unsigned int kb=Kh[(size_t)i*H + lane]; float ksc=__half2float(Ksh[i]);
        int k0=kb&0xF; if(k0>=8)k0-=16; int k1=(kb>>4)&0xF; if(k1>=8)k1-=16;
        float partial=q0*((float)k0*ksc)+q1*((float)k1*ksc);
        #pragma unroll
        for(int off=16;off>0;off>>=1) partial+=__shfl_down_sync(0xffffffff,partial,off);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;
        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p;
        unsigned int vb=Vh[(size_t)i*H + lane]; float vsc=__half2float(Vsh[i]);
        int v0=vb&0xF; if(v0>=8)v0-=16; int v1=(vb>>4)&0xF; if(v1>=8)v1-=16;
        o0=beta*o0+p*((float)v0*vsc); o1=beta*o1+p*((float)v1*vsc);
        m_run=m_new;
    }
    __shared__ float sm[W],sl[W],so[W][64];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    so[warp][2*lane+0]=o0; so[warp][2*lane+1]=o1;
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,c0=0.0f,c1=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float m_w=sm[w],m_new=fmaxf(m_c,m_w);
            float a=__expf(m_c-m_new),b=__expf(m_w-m_new); l_c=a*l_c+b*sl[w];
            c0=a*c0+b*so[w][2*lane+0]; c1=a*c1+b*so[w][2*lane+1]; m_c=m_new; }
        oh[2*lane+0]=__float2half(c0/l_c); oh[2*lane+1]=__float2half(c1/l_c);
    }
}
torch::Tensor decode_attn_int4(torch::Tensor q, torch::Tensor K, torch::Tensor V,
    torch::Tensor Ks, torch::Tensor Vs, int64_t W){
    TORCH_CHECK(K.scalar_type()==torch::kByte&&V.scalar_type()==torch::kByte,"packed uint8");
    const int B=K.size(0),N=K.size(1),H=K.size(2); TORCH_CHECK(H==32,"D=64 packed -> 32 bytes");
    auto o=torch::empty({B,64},q.options()); const float scale=1.0f/sqrtf(64.0f);
    dim3 grid(B),block(32*W);
    #define L(Wv) decode_attn_int4_kernel<Wv><<<grid,block>>>((const __half*)q.data_ptr<at::Half>(),(const uint8_t*)K.data_ptr<uint8_t>(),(const uint8_t*)V.data_ptr<uint8_t>(),(const __half*)Ks.data_ptr<at::Half>(),(const __half*)Vs.data_ptr<at::Half>(),(__half*)o.data_ptr<at::Half>(),N,scale)
    switch(W){case 2:L(2);break;case 4:L(4);break;case 8:L(8);break;default:TORCH_CHECK(false,"W");}
    #undef L
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"launch"); return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_int4",&decode_attn_int4,"int4 KV");}
"""
open("kernels/decode_attn_int4.cu","w").write(int4_src)
mod_int4 = load(name="decode_attn_int4", sources=["kernels/decode_attn_int4.cu"], extra_cuda_cflags=["-O3"], verbose=True)
print("int4 kernel compiled OK")


int4 correctness gate + accuracy vs true fp32

In [ ]:

torch.manual_seed(0)
def check_int4(B,N,W,atol=6e-2):
    q=torch.randn(B,64,dtype=torch.float16,device=dev).contiguous()
    K=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
    V=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
    Kp,Ks=quant_int4_per_token(K.cpu()); Vp,Vs=quant_int4_per_token(V.cpu())
    Kp,Ks,Vp,Vs=Kp.to(dev),Ks.to(dev),Vp.to(dev),Vs.to(dev)
    out=mod_int4.decode_attn_int4(q,Kp,Vp,Ks,Vs,W)
    Kd=dequant_int4(Kp.cpu(),Ks.cpu()).to(dev).half(); Vd=dequant_int4(Vp.cpu(),Vs.cpu()).to(dev).half()
    ref=F.scaled_dot_product_attention(q.view(B,1,1,64),Kd.view(B,1,N,64),Vd.view(B,1,N,64)).view(B,64)
    d=(out-ref).abs().max().item(); ok=torch.allclose(out,ref,atol=atol)
    print(f"W={W} B={B:4d} N={N:5d} | kernel-vs-dequant-ref diff {d:.3e} | {'PASS' if ok else 'FAIL'}")
    return ok
ok=True
for W in [2,4,8]:
    for (B,N) in [(1,1024),(8,2048),(256,4096)]: ok&=check_int4(B,N,W)
    ok&=check_int4(4,3,W)
print("\nALL PASS" if ok else "\nFAILED — stop"); assert ok

# accuracy vs TRUE fp32
print("\nint4 end-to-end accuracy vs true fp32:")
for (B,N) in [(8,1024),(8,4096),(64,2048)]:
    q=torch.randn(B,64,dtype=torch.float16,device=dev).contiguous()
    K=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
    V=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
    Kp,Ks=quant_int4_per_token(K.cpu()); Vp,Vs=quant_int4_per_token(V.cpu())
    out=mod_int4.decode_attn_int4(q,Kp.to(dev),Vp.to(dev),Ks.to(dev),Vs.to(dev),4).float()
    ref=torch.softmax(torch.einsum('bnd,bd->bn',K.float(),q.float())/8.0,1)
    ref=torch.einsum('bn,bnd->bd',ref,V.float())
    rel=(out-ref).abs().max().item()/ref.abs().max().item()
    print(f"B={B} N={N} | int4-vs-fp32 rel {rel:.4e}")


W=2 B=   1 N= 1024 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=2 B=   8 N= 2048 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=2 B= 256 N= 4096 | kernel-vs-dequant-ref diff 1.221e-04 | PASS
W=2 B=   4 N=    3 | kernel-vs-dequant-ref diff 1.953e-03 | PASS
W=4 B=   1 N= 1024 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=4 B=   8 N= 2048 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=4 B= 256 N= 4096 | kernel-vs-dequant-ref diff 6.866e-05 | PASS
W=4 B=   4 N=    3 | kernel-vs-dequant-ref diff 9.766e-04 | PASS
W=8 B=   1 N= 1024 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=8 B=   8 N= 2048 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=8 B= 256 N= 4096 | kernel-vs-dequant-ref diff 6.104e-05 | PASS
W=8 B=   4 N=    3 | kernel-vs-dequant-ref diff 9.766e-04 | PASS

ALL PASS

int4 end-to-end accuracy vs true fp32:
B=8 N=1024 | int4-vs-fp32 rel 1.4545e-01
B=8 N=4096 | int4-vs-fp32 rel 2.2689e-01
B=64 N=2048 | int4-vs-fp32 rel 1.7994e-01


Standalone benchmark: fp16 vs int8 vs int4 (wall-clock ms, measured). Feeds results.json.

In [ ]:

torch.manual_seed(0)
def bench(fn,iters=100,warmup=30):
    warm=torch.randn(2048,2048,device=dev,dtype=torch.float16)
    for _ in range(30): _=warm@warm
    torch.cuda.synchronize()
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(iters):
        s.record(); fn(); e.record(); torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

peak=1555e9; rows=[]
print(f"{'B':>5} {'N':>6} {'W':>3} {'fp16':>8} {'int8':>8} {'int4':>8} {'i8/f16':>7} {'i4/f16':>7}")
for B in [8,256,2048]:
    for W in [8]:
        N=4096
        q=torch.randn(B,64,dtype=torch.float16,device=dev).contiguous()
        K=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
        V=torch.randn(B,N,64,dtype=torch.float16,device=dev).contiguous()
        Kq,Ks=qpt(K); Vq,Vs=qpt(V)
        Kp,Ks4=quant_int4_per_token(K.cpu()); Vp,Vs4=quant_int4_per_token(V.cpu())
        Kp,Vp,Ks4,Vs4=Kp.to(dev),Vp.to(dev),Ks4.to(dev),Vs4.to(dev)
        t16=bench(lambda: mod_fp16.decode_attn_fp16_mw(q,K,V,W))
        t8 =bench(lambda: mod_int8.decode_attn_int8_mw(q,Kq,Vq,Ks,Vs,W))
        t4 =bench(lambda: mod_int4.decode_attn_int4(q,Kp,Vp,Ks4,Vs4,W))
        print(f"{B:>5} {N:>6} {W:>3} {t16:>8.4f} {t8:>8.4f} {t4:>8.4f} {t16/t8:>6.2f}x {t16/t4:>6.2f}x")
        rows.append(dict(B=B,N=N,W=W,fp16_ms=t16,int8_ms=t8,int4_ms=t4,int8_speedup=t16/t8,int4_speedup=t16/t4))
json.dump(dict(peak_GBs=1555.0, note="wall-clock ms; speedup vs fp16 fused; KV footprint fp16:int8:int4 = 4:2:1", rows=rows),
          open("benchmarks/standalone_quant.json","w"), indent=2)
print("\nsaved benchmarks/standalone_quant.json")


    B      N   W     fp16     int8     int4  i8/f16  i4/f16
    8   4096   8   0.2263   0.2478   0.2662   0.91x   0.85x
  256   4096   8   0.4792   0.4864   0.4915   0.99x   0.98x
 2048   4096   8   2.0951   1.6814   1.5585   1.25x   1.34x

saved benchmarks/standalone_quant.json


GQA-aware kernel (investigate)

Query head h reads KV head h//group without materializing repeat_kv, the potential win is
better L2 reuse (4 query heads hit the same KV lines), not raw bandwidth (we already showed at
scale the kernel is latency-bound, not byte-bound). Prediction: <1.1x, possibly nothing. We measure.

In [ ]:

gqa_src = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cuda_runtime.h>
// GQA-aware fp16: grid = B*Hq blocks; block maps to (b,h); reads KV head h/G directly.
// q:[B,Hq,D]  K,V:[B,Hkv,N,D]  o:[B,Hq,D]
template <int D, int W>
__global__ void decode_attn_gqa_kernel(
    const __half* __restrict__ q, const __half* __restrict__ K, const __half* __restrict__ V,
    __half* __restrict__ o, int N, int Hq, int Hkv, int G, float scale){
    const int blk=blockIdx.x;            // 0 .. B*Hq-1
    const int b=blk/Hq, h=blk%Hq, kvh=h/G;
    const int lane=threadIdx.x&31, warp=threadIdx.x>>5, ELEMS=D/32;
    const __half* qh=q+((size_t)b*Hq + h)*D;
    const __half* Kh=K+(((size_t)b*Hkv + kvh)*N)*D;
    const __half* Vh=V+(((size_t)b*Hkv + kvh)*N)*D;
    __half* oh=o+((size_t)b*Hq + h)*D;
    float q_reg[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) q_reg[e]=__half2float(qh[lane+e*32]);
    float m_run=-INFINITY,l_run=0.0f,o_run[ELEMS];
    #pragma unroll
    for(int e=0;e<ELEMS;++e) o_run[e]=0.0f;
    for(int i=warp;i<N;i+=W){
        const __half* Ki=Kh+(size_t)i*D; float partial=0.0f;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) partial+=q_reg[e]*__half2float(Ki[lane+e*32]);
        #pragma unroll
        for(int off=16;off>0;off>>=1) partial+=__shfl_down_sync(0xffffffff,partial,off);
        float s=__shfl_sync(0xffffffff,partial,0)*scale;
        float m_new=fmaxf(m_run,s), beta=__expf(m_run-m_new), p=__expf(s-m_new);
        l_run=beta*l_run+p; const __half* Vi=Vh+(size_t)i*D;
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_run[e]=beta*o_run[e]+p*__half2float(Vi[lane+e*32]);
        m_run=m_new;
    }
    __shared__ float sm[W],sl[W],so[W][D];
    if(lane==0){sm[warp]=m_run;sl[warp]=l_run;}
    #pragma unroll
    for(int e=0;e<ELEMS;++e) so[warp][lane+e*32]=o_run[e];
    __syncthreads();
    if(warp==0){
        float m_c=-INFINITY,l_c=0.0f,o_c[ELEMS];
        #pragma unroll
        for(int e=0;e<ELEMS;++e) o_c[e]=0.0f;
        #pragma unroll
        for(int w=0;w<W;++w){ float m_w=sm[w],m_new=fmaxf(m_c,m_w);
            float a=__expf(m_c-m_new),b2=__expf(m_w-m_new); l_c=a*l_c+b2*sl[w];
            #pragma unroll
            for(int e=0;e<ELEMS;++e) o_c[e]=a*o_c[e]+b2*so[w][lane+e*32]; m_c=m_new; }
        #pragma unroll
        for(int e=0;e<ELEMS;++e) oh[lane+e*32]=__float2half(o_c[e]/l_c);
    }
}
torch::Tensor decode_attn_gqa(torch::Tensor q, torch::Tensor K, torch::Tensor V, int64_t W){
    // q:[B,Hq,D] K,V:[B,Hkv,N,D]
    const int B=q.size(0),Hq=q.size(1),D=q.size(2);
    const int Hkv=K.size(1),N=K.size(2); const int G=Hq/Hkv;
    auto o=torch::empty({B,Hq,D},q.options()); const float scale=1.0f/sqrtf((float)D);
    dim3 grid(B*Hq),block(32*W);
    #define L(Dv,Wv) decode_attn_gqa_kernel<Dv,Wv><<<grid,block>>>((const __half*)q.data_ptr<at::Half>(),(const __half*)K.data_ptr<at::Half>(),(const __half*)V.data_ptr<at::Half>(),(__half*)o.data_ptr<at::Half>(),N,Hq,Hkv,G,scale)
    TORCH_CHECK(D==64||D==128,"D");
    if(D==64){switch(W){case 2:L(64,2);break;case 4:L(64,4);break;case 8:L(64,8);break;default:TORCH_CHECK(false,"W");}}
    else{switch(W){case 2:L(128,2);break;case 4:L(128,4);break;case 8:L(128,8);break;default:TORCH_CHECK(false,"W");}}
    #undef L
    TORCH_CHECK(cudaGetLastError()==cudaSuccess,"launch"); return o;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME,m){m.def("decode_attn_gqa",&decode_attn_gqa,"GQA-aware fp16");}
"""
open("kernels/decode_attn_gqa.cu","w").write(gqa_src)
mod_gqa = load(name="decode_attn_gqa", sources=["kernels/decode_attn_gqa.cu"], extra_cuda_cflags=["-O3"], verbose=True)
print("GQA-aware kernel compiled OK")

# correctness: vs expand-then-SDPA
torch.manual_seed(0)
def check_gqa(B,Hq,Hkv,N,W,atol=2e-2):
    D=64; G=Hq//Hkv
    q=torch.randn(B,Hq,D,dtype=torch.float16,device=dev).contiguous()
    K=torch.randn(B,Hkv,N,D,dtype=torch.float16,device=dev).contiguous()
    V=torch.randn(B,Hkv,N,D,dtype=torch.float16,device=dev).contiguous()
    out=mod_gqa.decode_attn_gqa(q,K,V,W)               # [B,Hq,D]
    Ke=K.repeat_interleave(G,dim=1); Ve=V.repeat_interleave(G,dim=1)  # [B,Hq,N,D]
    ref=F.scaled_dot_product_attention(q.view(B,Hq,1,D),Ke,Ve).view(B,Hq,D)
    d=(out-ref).abs().max().item(); ok=torch.allclose(out,ref,atol=atol)
    print(f"B={B} Hq={Hq} Hkv={Hkv} N={N} W={W} | diff {d:.3e} | {'PASS' if ok else 'FAIL'}")
    return ok
ok=True
for W in [4,8]:
    for (B,N) in [(1,1024),(8,2048),(16,4096)]:
        ok&=check_gqa(B,32,8,N,W)   # Llama-3.2-1B geometry: 32 q / 8 kv
print("\nALL PASS" if ok else "\nFAILED"); assert ok

# benchmark: GQA-aware (no expansion) vs expand-then-fp16-kernel, real geometry
print("\nGQA-aware vs expand-then-kernel (Llama geometry 32q/8kv):")
rows=[]
for B in [1,4,16]:
    for W in [8]:
        N=4096; D=64; Hq,Hkv,G=32,8,4
        q=torch.randn(B,Hq,D,dtype=torch.float16,device=dev).contiguous()
        K=torch.randn(B,Hkv,N,D,dtype=torch.float16,device=dev).contiguous()
        V=torch.randn(B,Hkv,N,D,dtype=torch.float16,device=dev).contiguous()
        def gqa_call():
            return mod_gqa.decode_attn_gqa(q,K,V,W)
        def expand_call():
            Ke=K.repeat_interleave(G,dim=1).reshape(B*Hq,N,D).contiguous()
            Ve=V.repeat_interleave(G,dim=1).reshape(B*Hq,N,D).contiguous()
            q2=q.reshape(B*Hq,D).contiguous()
            return mod_fp16.decode_attn_fp16_mw(q2,Ke,Ve,W)
        tg=bench(gqa_call); te=bench(expand_call)
        print(f"B={B:3d} | gqa-aware {tg:.4f} ms | expand+kernel {te:.4f} ms | gqa/expand {te/tg:.2f}x")
        rows.append(dict(B=B,W=W,gqa_aware_ms=tg,expand_kernel_ms=te,speedup=te/tg))
json.dump(dict(rows=rows, note="GQA-aware avoids repeat_kv materialization"), open("benchmarks/gqa_aware.json","w"), indent=2)
print("saved benchmarks/gqa_aware.json")


GQA-aware kernel compiled OK
B=1 Hq=32 Hkv=8 N=1024 W=4 | diff 1.221e-04 | PASS
B=8 Hq=32 Hkv=8 N=2048 W=4 | diff 6.104e-05 | PASS
B=16 Hq=32 Hkv=8 N=4096 W=4 | diff 6.104e-05 | PASS
B=1 Hq=32 Hkv=8 N=1024 W=8 | diff 1.221e-04 | PASS
B=8 Hq=32 Hkv=8 N=2048 W=8 | diff 1.221e-04 | PASS
B=16 Hq=32 Hkv=8 N=4096 W=8 | diff 6.104e-05 | PASS

ALL PASS

GQA-aware vs expand-then-kernel (Llama geometry 32q/8kv):
B=  1 | gqa-aware 0.2417 ms | expand+kernel 0.5898 ms | gqa/expand 2.44x
B=  4 | gqa-aware 0.4588 ms | expand+kernel 0.8735 ms | gqa/expand 1.90x
B= 16 | gqa-aware 0.4506 ms | expand+kernel 1.6148 ms | gqa/expand 3.58x
saved benchmarks/gqa_aware.json


Live model integration

Below: load Llama-3.2-1B, then inspect the exact attention API.

Then register
a custom decode-attention that calls our GQA-aware fp16 kernel (dense SDPA kept for prefill; RoPE already
applied upstream; GQA handled by the kernel's head mapping), verify generated tokens match stock, and
measure decode throughput.

## 10. Load model + INSPECT the attention API (always runs; print & read).

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer
import inspect, transformers

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(dev).eval()
cfg = model.config
print("transformers:", transformers.__version__)
print("layers:", cfg.num_hidden_layers, "| n_heads:", cfg.num_attention_heads,
      "| n_kv:", cfg.num_key_value_heads,
      "| head_dim:", getattr(cfg,"head_dim",cfg.hidden_size//cfg.num_attention_heads))
attn0 = model.model.layers[0].self_attn
print("attn class:", type(attn0).__name__)
print("attn fwd sig:", inspect.signature(attn0.forward))
print("cfg._attn_implementation:", getattr(cfg,"_attn_implementation",None))
try:
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
    print("ALL_ATTENTION_FUNCTIONS keys:", list(ALL_ATTENTION_FUNCTIONS.keys()))
    HAVE_REG=True
except Exception as ex:
    print("no ALL_ATTENTION_FUNCTIONS:", ex); HAVE_REG=False
try:
    from transformers.integrations.sdpa_attention import sdpa_attention_forward
    print("sdpa_attention_forward import: OK")
except Exception as ex:
    print("sdpa_attention_forward import FAILED:", ex)


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

transformers: 5.16.1
layers: 16 | n_heads: 32 | n_kv: 8 | head_dim: 64
attn class: LlamaAttention
attn fwd sig: (hidden_states: torch.Tensor, position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None, attention_mask: torch.Tensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]) -> tuple[torch.Tensor, torch.Tensor]
cfg._attn_implementation: sdpa
ALL_ATTENTION_FUNCTIONS keys: ['flash_attention_4', 'flash_attention_3', 'flash_attention_2', 'flex_attention', 'sdpa', 'paged|flash_attention_4', 'paged|flash_attention_3', 'paged|flash_attention_2', 'paged|sdpa', 'paged|eager']
sdpa_attention_forward import: OK


Register custom decode-attention (fp16 GQA-aware kernel; SDPA for prefill)

The custom function receives already-RoPE'd Q,K,V as [B,H,S,D] (K,V with n_kv heads). We use SDPA for
prefill (q_len>1) and our kernel for decode (q_len==1). This is the supported extension point.

In [ ]:

W_LIVE = 8
try:
    from transformers.integrations.sdpa_attention import sdpa_attention_forward as _sdpa_fwd
except Exception:
    _sdpa_fwd = None

def kernel_attention_forward(module, query, key, value, attention_mask=None, scaling=None, dropout=0.0, **kwargs):
    # query:[B,Hq,q,D]  key/value:[B,Hkv,kv,D]
    q_len = query.shape[2]
    if q_len != 1:
        if _sdpa_fwd is not None:
            return _sdpa_fwd(module, query, key, value, attention_mask, scaling=scaling, dropout=dropout, **kwargs)
        # manual SDPA fallback (prefill)
        B,Hq,S,D = query.shape; Hkv=key.shape[1]; G=Hq//Hkv
        k=key.repeat_interleave(G,1); v=value.repeat_interleave(G,1)
        out=F.scaled_dot_product_attention(query,k,v,is_causal=True,scale=scaling)
        return out.transpose(1,2).contiguous(), None
    # decode path -> our GQA-aware kernel
    B,Hq,_,D = query.shape
    q2 = query[:,:,0,:].contiguous().half()          # [B,Hq,D]
    K  = key.contiguous().half()                     # [B,Hkv,kv,D]
    V  = value.contiguous().half()
    out = mod_gqa.decode_attn_gqa(q2, K, V, W_LIVE)  # [B,Hq,D]
    out = out.unsqueeze(2)                            # [B,Hq,1,D]
    return out.transpose(1,2).contiguous(), None     # [B,1,Hq,D]

if HAVE_REG:
    ALL_ATTENTION_FUNCTIONS["kernel_decode"] = kernel_attention_forward
    print("registered 'kernel_decode'")
else:
    print("registry unavailable — integration path will need adjustment (paste S10 output).")


registered 'kernel_decode'


In [ ]:

prompt = "The theory of general relativity states that"
ids = tok(prompt, return_tensors="pt").to(dev)

def generate(attn_impl, n=40):
    model.config._attn_implementation = attn_impl
    for lyr in model.model.layers:            # some versions cache per-module
        try: lyr.self_attn.config._attn_implementation = attn_impl
        except Exception: pass
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False,
                             use_cache=True, pad_token_id=tok.eos_token_id)
    return out[0].tolist()

base = generate("sdpa")
try:
    ours = generate("kernel_decode")
    match = base == ours
    print("stock tokens :", base[-40:])
    print("kernel tokens:", ours[-40:])
    print("\nEXACT TOKEN MATCH:", match)
    if not match:
        # show first divergence
        for i,(a,b) in enumerate(zip(base,ours)):
            if a!=b: print("first divergence at position", i, "stock",a,"ours",b); break
    print("\nstock text :", tok.decode(base, skip_special_tokens=True))
    print("kernel text:", tok.decode(ours, skip_special_tokens=True))
except Exception as ex:
    import traceback; traceback.print_exc()
    print("\nINTEGRATION ERROR above — paste S10 output + this traceback.")
finally:
    model.config._attn_implementation = "sdpa"


[transformers] Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


stock tokens : [279, 71019, 2115, 374, 264, 83233, 315, 3634, 7394, 13, 578, 83233, 315, 3634, 7394, 374, 264, 29774, 315, 279, 2144, 430, 279, 71019, 2115, 374, 264, 29774, 315, 279, 2144, 430, 279, 71019, 2115, 374, 264, 29774, 315, 279]
kernel tokens: [279, 71019, 2115, 374, 264, 83233, 315, 3634, 7394, 13, 578, 83233, 315, 3634, 7394, 374, 264, 29774, 315, 279, 2144, 430, 279, 71019, 2115, 374, 264, 29774, 315, 279, 2144, 430, 279, 71019, 2115, 374, 264, 29774, 315, 279]

EXACT TOKEN MATCH: True

stock text : The theory of general relativity states that the gravitational field is a curvature of space-time. The curvature of space-time is a consequence of the fact that the gravitational field is a consequence of the fact that the gravitational field is a consequence of the
kernel text: The theory of general relativity states that the gravitational field is a curvature of space-time. The curvature of space-time is a consequence of the fact that the gravitational field is a consequence

End to end decode throughput: stock SDPA vs kernel (measured tokens/sec).

In [ ]:

def decode_tps(attn_impl, prompt_len=512, gen=128, reps=3):
    model.config._attn_implementation = attn_impl
    for lyr in model.model.layers:
        try: lyr.self_attn.config._attn_implementation = attn_impl
        except Exception: pass
    ids = torch.randint(0, cfg.vocab_size, (1, prompt_len), device=dev)
    best = 1e9
    with torch.no_grad():
        for _ in range(2):  # warmup
            model.generate(ids, max_new_tokens=8, do_sample=False, use_cache=True, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        for _ in range(reps):
            t0=time.perf_counter()
            model.generate(ids, max_new_tokens=gen, do_sample=False, use_cache=True, pad_token_id=tok.eos_token_id)
            torch.cuda.synchronize()
            best=min(best, time.perf_counter()-t0)
    return gen/best

try:
    tps_sdpa = decode_tps("sdpa")
    tps_ker  = decode_tps("kernel_decode")
    print(f"stock SDPA : {tps_sdpa:7.2f} tok/s")
    print(f"kernel     : {tps_ker:7.2f} tok/s")
    print(f"speedup    : {tps_ker/tps_sdpa:.3f}x")
    json.dump(dict(prompt_len=512, gen=128, sdpa_tps=tps_sdpa, kernel_tps=tps_ker,
                   speedup=tps_ker/tps_sdpa), open("benchmarks/end_to_end.json","w"), indent=2)
    print("saved benchmarks/end_to_end.json")
except Exception as ex:
    import traceback; traceback.print_exc()
finally:
    model.config._attn_implementation="sdpa"


[transformers] Both `max_new_tokens` (=8) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=8) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

stock SDPA :   51.89 tok/s
kernel     :   54.68 tok/s
speedup    : 1.054x
saved benchmarks/end_to_end.json


In [ ]:
# decode-ONLY timing: fixed KV length, time single forward passes (no prefill in the measurement)
@torch.no_grad()
def decode_only_ms(attn_impl, ctx=1024, reps=200):
    model.config._attn_implementation = attn_impl
    for lyr in model.model.layers:
        try: lyr.self_attn.config._attn_implementation = attn_impl
        except Exception: pass
    ids = torch.randint(0, cfg.vocab_size, (1, ctx), device=dev)
    out = model(ids, use_cache=True)          # build a ctx-length cache once
    past = out.past_key_values
    step = torch.randint(0, cfg.vocab_size, (1, 1), device=dev)
    for _ in range(20): model(step, past_key_values=past, use_cache=True)  # warmup
    torch.cuda.synchronize()
    s,e=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True); ts=[]
    for _ in range(reps):
        s.record(); model(step, past_key_values=past, use_cache=True); e.record()
        torch.cuda.synchronize(); ts.append(s.elapsed_time(e))
    ts.sort(); return ts[len(ts)//2]

try:
    for ctx in [512, 2048]:
        d_sdpa = decode_only_ms("sdpa", ctx)
        d_ker  = decode_only_ms("kernel_decode", ctx)
        print(f"ctx={ctx:5d} | SDPA {d_sdpa:.4f} ms | kernel {d_ker:.4f} ms | speedup {d_sdpa/d_ker:.3f}x")
finally:
    model.config._attn_implementation="sdpa"

ctx=  512 | SDPA 17.9456 ms | kernel 17.2636 ms | speedup 1.040x
ctx= 2048 | SDPA 17.7951 ms | kernel 17.3681 ms | speedup 1.025x


In [ ]:

import glob, json
results = {}
for f in sorted(glob.glob("benchmarks/*.json")):
    key = os.path.splitext(os.path.basename(f))[0]
    results[key] = json.load(open(f))
json.dump(results, open("results.json","w"), indent=2)
print("wrote results.json with sections:", list(results.keys()))


wrote results.json with sections: ['end_to_end', 'gqa_aware', 'standalone_quant']


In [ ]:

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, json
R=json.load(open("results.json"))
# quant speedups at largest B
sq=R.get("standalone_quant",{}).get("rows",[])
if sq:
    big=[r for r in sq if r["B"]==max(x["B"] for x in sq)]
    r=big[0]
    fig,ax=plt.subplots(figsize=(6,4))
    ax.bar(["fp16","int8","int4"],[1.0, r["int8_speedup"], r["int4_speedup"]],
           color=["#888","#3b7","#37b"])
    ax.set_ylabel("speedup vs fp16 fused"); ax.set_title(f"KV-precision speedup (B={r['B']}, N={r['N']})")
    for i,v in enumerate([1.0,r["int8_speedup"],r["int4_speedup"]]): ax.text(i,v+0.01,f"{v:.2f}x",ha="center")
    fig.tight_layout(); fig.savefig("plots/quant_speedup.png",dpi=130); print("saved plots/quant_speedup.png")
ee=R.get("end_to_end")
if ee:
    fig,ax=plt.subplots(figsize=(5,4))
    ax.bar(["stock SDPA","our kernel"],[ee["sdpa_tps"],ee["kernel_tps"]],color=["#888","#3b7"])
    ax.set_ylabel("decode tokens/sec"); ax.set_title("End-to-end decode (Llama-3.2-1B)")
    ax.text(2, r["int4_speedup"]/2, "~18% err\nnot usable", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    fig.tight_layout(); fig.savefig("plots/end_to_end.png",dpi=130); print("saved plots/end_to_end.png")
print("plots done")


saved plots/quant_speedup.png
saved plots/end_to_end.png
plots done


pytest correctness suite

In [ ]:
%%writefile tests/test_kernels.py
import pytest, torch, math
import torch.nn.functional as F
from torch.utils.cpp_extension import load
pytestmark = pytest.mark.skipif(not torch.cuda.is_available(), reason="CUDA only")

@pytest.fixture(scope="module")
def mods():
    fp16=load(name="t_fp16",sources=["kernels/decode_attn_fp16_mw.cu"],extra_cuda_cflags=["-O3"],verbose=False)
    int8=load(name="t_int8",sources=["kernels/decode_attn_int8_mw.cu"],extra_cuda_cflags=["-O3"],verbose=False)
    int4=load(name="t_int4",sources=["kernels/decode_attn_int4.cu"],extra_cuda_cflags=["-O3"],verbose=False)
    gqa =load(name="t_gqa", sources=["kernels/decode_attn_gqa.cu"], extra_cuda_cflags=["-O3"],verbose=False)
    return fp16,int8,int4,gqa

def _sdpa(q,K,V):
    B,D=q.shape;N=K.shape[1]
    return F.scaled_dot_product_attention(q.view(B,1,1,D),K.view(B,1,N,D),V.view(B,1,N,D)).view(B,D)

@pytest.mark.parametrize("B,N,W",[(1,1024,8),(8,2048,4),(256,4096,8)])
def test_fp16(mods,B,N,W):
    fp16,_,_,_=mods; dev="cuda"; torch.manual_seed(0)
    q=torch.randn(B,64,dtype=torch.float16,device=dev)
    K=torch.randn(B,N,64,dtype=torch.float16,device=dev)
    V=torch.randn(B,N,64,dtype=torch.float16,device=dev)
    assert torch.allclose(fp16.decode_attn_fp16_mw(q,K,V,W),_sdpa(q,K,V),atol=2e-2)

@pytest.mark.parametrize("B,N,W",[(1,1024,8),(8,2048,4)])
def test_int8(mods,B,N,W):
    _,int8,_,_=mods; dev="cuda"; torch.manual_seed(0)
    q=torch.randn(B,64,dtype=torch.float16,device=dev)
    K=torch.randn(B,N,64,dtype=torch.float16,device=dev)
    V=torch.randn(B,N,64,dtype=torch.float16,device=dev)
    def qpt(X):
        a=X.abs().amax(2,keepdim=True);s=(a/127.0).clamp(min=1e-8)
        return torch.clamp(torch.round(X/s),-127,127).to(torch.int8).contiguous(),s.squeeze(2).half().contiguous()
    Kq,Ks=qpt(K);Vq,Vs=qpt(V)
    out=int8.decode_attn_int8_mw(q,Kq,Vq,Ks,Vs,W)
    Kd=(Kq.float()*Ks.float()[:,:,None]).half();Vd=(Vq.float()*Vs.float()[:,:,None]).half()
    ref=F.scaled_dot_product_attention(q.view(B,1,1,64),Kd.view(B,1,N,64),Vd.view(B,1,N,64)).view(B,64)
    assert torch.allclose(out,ref,atol=3e-2)

@pytest.mark.parametrize("B,N,W",[(1,1024,8),(16,4096,8)])
def test_gqa(mods,B,N,W):
    _,_,_,gqa=mods; dev="cuda"; torch.manual_seed(0)
    Hq,Hkv,D=32,8,64;G=Hq//Hkv
    q=torch.randn(B,Hq,D,dtype=torch.float16,device=dev)
    K=torch.randn(B,Hkv,N,D,dtype=torch.float16,device=dev)
    V=torch.randn(B,Hkv,N,D,dtype=torch.float16,device=dev)
    out=gqa.decode_attn_gqa(q,K,V,W)
    Ke=K.repeat_interleave(G,1);Ve=V.repeat_interleave(G,1)
    ref=F.scaled_dot_product_attention(q.view(B,Hq,1,D),Ke,Ve).view(B,Hq,D)
    assert torch.allclose(out,ref,atol=2e-2)


Writing tests/test_kernels.py


In [ ]:
!cd /content && python -m pytest tests/test_kernels.py -q 2>&1 | tail -n 20

.......                                                                  [100%]
7 passed in 2.28s
